Exploration Charts — GoEmotions (28 fixed labels) fork
==========================================================
Charts the GoEmotions (28 fixed labels) scores produced by `05.2_song_analysis_goemotions.ipynb`.

The companion notebook `06.1_exploration_charts_zeroshot.ipynb` runs the identical chart set over the other
classifier. Both forks are produced by `04_classification.ipynb` under one **shared
scoring contract** (see its header), so they differ only where they must:

| | zero-shot NLI (04 §4A → 05.1 → 06.1) | GoEmotions (04 §4B → 05.2 → 06.2) |
|---|---|---|
| scoring | independent per-label, [0, 1] | *same* |
| `unclassified` | no scoreable lyrics | *same* — drops the same ~107 songs |
| confidence | flagged at 0.30, never dropped | *same* |
| **labels** | **10, hand-picked for songs** | **28, fixed (Reddit-trained)** |
| **includes `neutral`** | **no** | **yes** |
| **model** | **`bart-large-mnli`, zero-shot** | **`roberta-base-go_emotions`, supervised** |
| **typical top score** | **~0.97** | **~0.50** |

The bold rows are the real, irreducible differences; everything above them used
to differ too, purely by config. The last row still matters for reading charts:
RoBERTa's sigmoids are calibrated systematically lower than bart-mnli's
entailment probabilities, so **a raw 0.4 does not mean the same thing in each
fork** even under the shared contract.

Absolute-magnitude charts below are therefore labelled fork-local. The **z-score
charts (§ 4) are the ones that survive a cross-fork read** — standardising
within a fork cancels the calibration offset and leaves only "which regions are
unusually high on this emotion, relative to this fork's own baseline".

`neutral` is charted out of the emotion grids (it has no counterpart in the
zero-shot taxonomy and would otherwise dominate every panel) and is reported on
its own below. The dominant-emotion charts use `dominant_emotion_emotive` —
the strongest non-neutral label — computed in `05.2`.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
PROCESSED = PROJECT_ROOT / "data" / "processed"

# ── Fork config ───────────────────────────────────────────────────────────────
LABEL        = "GoEmotions (28 fixed labels)"
INPUT_PATH   = PROCESSED / "05.2_titles_emotion_scores_goemotions.csv"
DOMINANT_COL = "dominant_emotion_emotive"   # column charted as "the song's emotion"
EXCLUDE      = ["neutral"]          # labels held out of the emotion charts
TOP_N        = 12            # emotions charted; 28 labels is unreadable in a grid; the tail is near-zero

df = pd.read_csv(INPUT_PATH)

emotion_cols = [
    c for c in df.columns
    if c.startswith("emotion_") and c.replace("emotion_", "") not in EXCLUDE
]
# Charted subset: the strongest TOP_N emotions overall. With 28 labels a full
# grid is unreadable, so the tail is trimmed here rather than in every chart.
charted_cols = (
    df[emotion_cols].mean().sort_values(ascending=False).head(TOP_N).index.tolist()
)
charted_names = [c.replace("emotion_", "") for c in charted_cols]

print(f"{LABEL}: {len(df)} songs, {len(emotion_cols)} emotion columns "
      f"(excluding {EXCLUDE or 'none'}), charting top {len(charted_cols)}")
print(charted_names)
display(df.head())

GoEmotions (28 fixed labels): 1471 songs, 27 emotion columns (excluding ['neutral']), charting top 12
['love', 'sadness', 'desire', 'disappointment', 'approval', 'curiosity', 'annoyance', 'amusement', 'admiration', 'confusion', 'joy', 'disapproval']


,rank,artist,title,region,spotify_uri,dominant_emotion,dominant_score,low_confidence,emotion_admiration,emotion_amusement,...,emotion_optimism,emotion_pride,emotion_realization,emotion_relief,emotion_remorse,emotion_sadness,emotion_surprise,emotion_neutral,dominant_emotion_emotive,neutral_score
0,1,"Mr Plata, El Americano 4KT",Las Muñequitas,Colombia,4nJJCRYru4QQakCiUA155f,amusement,0.3923,False,0.0040,0.3923,...,0.0035,0.0004,0.0075,0.0004,0.0006,0.0034,0.0011,0.3227,amusement,0.3227
1,2,"ARIA VEGA, Ryan Castro",CHÉVERE (premium_remix),Colombia,3CBEVPwR3kUXDoTx1lqFUQ,love,0.4801,False,0.0081,0.0181,...,0.0080,0.0004,0.0081,0.0010,0.0051,0.0304,0.0038,0.4012,love,0.4012
2,3,"Ryan Castro, Kapo, Gangsta",LA VILLA,Colombia,2ZyrAym0sRLwt4PhGotHuI,neutral,0.5830,False,0.0339,0.0371,...,0.0035,0.0018,0.0098,0.0016,0.0003,0.0011,0.0015,0.5830,love,0.5830
3,4,Kris R.,GANAS,Colombia,4KE9Ne3hgh18B3Th4xcylg,amusement,0.4549,False,0.0079,0.4549,...,0.0126,0.0007,0.0242,0.0028,0.0102,0.2587,0.0075,0.0365,amusement,0.0365
4,5,"W Sound, Beéle, Ovy On The Drums",La Plena - W Sound 05,Colombia,6iOndD4OFo7GkaDypWQIou,love,0.9290,False,0.0633,0.0044,...,0.0109,0.0009,0.0107,0.0009,0.0010,0.0022,0.0075,0.0312,love,0.0312


In [2]:
# ── Palette ───────────────────────────────────────────────────────────────────
# Sequential = ONE hue light→dark (magnitude). Diverging = two poles + a neutral
# gray midpoint (polarity around zero). Never a rainbow ramp for either job — a
# rainbow makes non-adjacent values look adjacent and hides the actual ordering.
SEQ_BLUE = [
    "#cde2fb", "#b7d3f6", "#9ec5f4", "#86b6ef", "#6da7ec",
    "#5598e7", "#3987e5", "#2a78d6", "#256abf", "#1c5cab",
    "#184f95", "#104281", "#0d366b",
]
DIVERGING = [[0.0, "#0d366b"], [0.25, "#3987e5"], [0.5, "#f0efec"],
             [0.75, "#e34948"], [1.0, "#7d1f1e"]]

# Shared z-score domain for § 4 / § 4b, hard-coded rather than taken from this
# fork's own max. Both forks' regional z-scores peak near ±2.4, so one fixed
# limit is what actually makes the two notebooks' z-charts stackable — a
# per-fork max would quietly give the same SD value a different colour and a
# different bar position in each. Widened past the data to leave label room.
Z_LIM = 2.9

# Fixed categorical order — hues are assigned by slot and never cycled or
# re-ordered by rank, so a given region keeps its colour across every chart.
CATEGORICAL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
               "#e87ba4", "#008300", "#4a3aa7", "#e34948"]

regions = sorted(df["region"].dropna().unique())
if len(regions) > len(CATEGORICAL):
    raise ValueError(
        f"{len(regions)} regions but only {len(CATEGORICAL)} validated hues. "
        "Fold the smallest into 'Other' or facet instead of generating a 9th hue."
    )
REGION_COLOR = dict(zip(regions, CATEGORICAL))

# `Global` is a reference playlist, not a market. § 4 / § 4b hold it out of the
# baseline and then score it against that baseline, so its position reads as
# "how Global sits relative to the markets" instead of it being averaged into
# the very mean it is compared with. Set to None to standardise across all
# regions instead.
REF_REGION = "Global" if "Global" in regions else None
baseline_regions = [r for r in regions if r != REF_REGION]
print(f"z baseline: {len(baseline_regions)} regions"
      + (f", holding out {REF_REGION!r}" if REF_REGION else " (no region held out)"))

LAYOUT = dict(
    template="plotly_white",
    font=dict(family="Inter, -apple-system, Helvetica, sans-serif", size=13,
              color="#0b0b0b"),
    title_font_size=17,
    margin=dict(l=90, r=40, t=90, b=70),
)
print(f"{len(regions)} regions: {regions}")

z baseline: 7 regions, holding out 'Global'
8 regions: ['Argentina', 'Colombia', 'Global', 'Japan', 'Singapore', 'Spain', 'Taiwan', 'USA']


### 1. Coverage and confidence

Only songs with no scoreable lyrics were dropped upstream in the 05.x join, and
both forks drop the same ~107 of them — so **this fork and its sibling are
charting the identical song set**, and per-region counts are directly
comparable between the two notebooks.

Low-confidence songs are *kept*, flagged rather than dropped. The flag uses the
same 0.30 bar in both forks, but because the two models are calibrated
differently the flagged *share* will not match — that difference is a real
property of the classifiers, which is exactly why it's shown here instead of
being silently absorbed into the row count.

In [3]:
region_counts = df["region"].value_counts().reindex(regions)

fig = go.Figure(go.Bar(
    x=region_counts.index,
    y=region_counts.values,
    marker=dict(color=[REGION_COLOR[r] for r in region_counts.index],
                cornerradius=4),
    text=region_counts.values,
    textposition="outside",
    hovertemplate="<b>%{x}</b><br>%{y} songs<extra></extra>",
    showlegend=False,
))
fig.update_layout(
    title=f"Classified songs per region — {LABEL}<br>"
          f"<sup>{len(df)} songs total; unclassified already removed in 05.x</sup>",
    yaxis_title="songs", xaxis_title=None, bargap=0.35,
    height=420, width=900, **LAYOUT,
)
fig.update_yaxes(gridcolor="#eceae5", zeroline=False)
fig.update_xaxes(showgrid=False)
fig.show()

# Confidence is carried as data, so report it rather than let it hide in the counts.
conf = (
    df.groupby("region")
      .agg(songs=("spotify_uri", "size"),
           median_dominant_score=("dominant_score", "median"),
           pct_low_confidence=("low_confidence", lambda s: s.mean() * 100))
      .reindex(regions)
)
print(f"Overall low_confidence share: {df['low_confidence'].mean():.1%} "
      f"(bar = 0.30, shared with the other fork)")
display(conf.round(2))

Overall low_confidence share: 10.1% (bar = 0.30, shared with the other fork)


,songs,median_dominant_score,pct_low_confidence
region,,,
Argentina,178,0.54,10.67
Colombia,188,0.51,7.45
Global,196,0.52,10.71
Japan,188,0.61,9.57
Singapore,183,0.54,12.57
Spain,193,0.47,9.33
Taiwan,149,0.71,8.72
USA,196,0.47,11.22


### 2. Dominant emotion mix

Which label wins outright per song, counted overall and split by region. Counts
are fork-local — the label sets don't line up, so a bar here has no counterpart
in the sibling notebook.

In [4]:
dom = df[DOMINANT_COL].value_counts()
dom = dom[~dom.index.isin(EXCLUDE)]

fig = go.Figure(go.Bar(
    x=dom.values, y=dom.index, orientation="h",
    marker=dict(color="#2a78d6", cornerradius=4),
    text=dom.values, textposition="outside",
    hovertemplate="<b>%{y}</b><br>%{x} songs<extra></extra>",
))
fig.update_layout(
    title=f"Dominant emotion across all songs — {LABEL}<br>"
          f"<sup>column charted: <code>{DOMINANT_COL}</code></sup>",
    xaxis_title="songs", yaxis=dict(autorange="reversed"),
    height=max(360, 26 * len(dom) + 140), width=860, bargap=0.35, **LAYOUT,
)
fig.update_xaxes(gridcolor="#eceae5", zeroline=False)
fig.update_yaxes(showgrid=False)
fig.show()

# Share of each region's songs, so a large region doesn't dominate by size alone
mix = (
    pd.crosstab(df["region"], df[DOMINANT_COL], normalize="index")
      .reindex(regions)
      .drop(columns=[c for c in EXCLUDE if c in df[DOMINANT_COL].unique()], errors="ignore")
)
mix = mix[dom.index[: min(12, len(dom))]]
display((mix * 100).round(1))

dominant_emotion_emotive,love,sadness,approval,curiosity,amusement,annoyance,desire,disappointment,admiration,disapproval,joy,confusion
region,,,,,,,,,,,,
Argentina,28.1,12.4,2.8,6.2,8.4,9.0,6.2,3.4,5.6,2.8,2.2,2.2
Colombia,37.2,14.9,1.6,4.3,16.5,5.9,2.1,2.7,4.8,2.7,1.1,2.1
Global,28.6,12.8,4.6,8.7,5.6,5.1,5.1,5.1,4.1,4.1,2.0,2.6
Japan,31.4,11.7,13.3,4.3,2.1,2.7,11.2,1.1,1.6,0.5,4.8,2.1
Singapore,26.8,10.4,13.1,11.5,3.3,5.5,8.2,6.6,2.7,2.7,2.2,0.0
Spain,30.1,11.4,3.6,5.7,9.8,6.2,7.3,4.7,3.1,2.1,1.0,4.1
Taiwan,12.1,6.7,36.9,8.7,3.4,4.7,5.4,4.7,2.7,3.4,2.7,0.7
USA,26.0,10.7,4.6,8.7,3.1,13.3,6.1,4.1,4.1,3.6,2.0,2.0


### 3. Regional emotion profile — raw mean scores *(fork-local)*

Mean score per region per emotion, on this fork's native scale. Useful for
reading *within* the grid (which emotion runs hottest in a region), **not** for
comparing a cell against the sibling notebook's same-named cell.

In [5]:
heat = df.groupby("region")[charted_cols].mean().reindex(regions)
heat.columns = charted_names

fig = px.imshow(
    heat.T,
    labels=dict(x="", y="", color="mean score"),
    color_continuous_scale=SEQ_BLUE,
    aspect="auto",
    text_auto=".2f",
)
fig.update_traces(
    textfont_size=11,
    hovertemplate="<b>%{x}</b> · %{y}<br>mean score %{z:.3f}<extra></extra>",
    xgap=2, ygap=2,   # surface gap between cells
)
fig.update_layout(
    title=f"Mean emotion score by region — {LABEL}<br>"
          f"<sup>Fork-local scale. Compare cells within this grid, not against 06.x's other fork.</sup>",
    height=60 + 34 * len(charted_names) + 140, width=1000,
    coloraxis_colorbar=dict(title="mean", thickness=12, len=0.7),
    **LAYOUT,
)
fig.show()
display(heat.round(3))

,love,sadness,desire,disappointment,approval,curiosity,annoyance,amusement,admiration,confusion,joy,disapproval
region,,,,,,,,,,,,
Argentina,0.219,0.101,0.065,0.078,0.046,0.050,0.050,0.051,0.050,0.035,0.027,0.030
Colombia,0.277,0.113,0.054,0.065,0.052,0.049,0.046,0.117,0.054,0.035,0.041,0.028
Global,0.219,0.099,0.072,0.079,0.059,0.062,0.054,0.041,0.038,0.036,0.031,0.031
Japan,0.289,0.131,0.136,0.070,0.069,0.081,0.033,0.018,0.031,0.038,0.055,0.026
Singapore,0.199,0.098,0.084,0.075,0.070,0.063,0.052,0.019,0.036,0.028,0.026,0.031
Spain,0.201,0.082,0.051,0.062,0.053,0.048,0.051,0.054,0.039,0.040,0.029,0.026
Taiwan,0.111,0.081,0.046,0.050,0.075,0.050,0.040,0.021,0.030,0.022,0.022,0.031
USA,0.186,0.085,0.072,0.076,0.059,0.065,0.079,0.024,0.035,0.037,0.027,0.035


### 4. Regional character — z-scored within this fork *(cross-fork readable)*

Each emotion column is standardised inside this fork as
`(region mean − baseline mean) / baseline std`, where the baseline is every
region **except `Global`**. Global is then scored against that baseline rather
than folded into it, so it works as a genuine reference point — "how the Global
playlist sits relative to the markets" — instead of being part of the mean it is
measured against. As a result the *markets* centre on zero; Global floats.

Standardising strips out both the overall scale and each label's own baseline
popularity, leaving a pure "how unusual is this region on this emotion" reading.

This is the chart to hold next to the sibling notebook's version of it. A value
of +1.5 means the same thing in both — *1.5 standard deviations above this
classifier's own market baseline* — even though the underlying raw scores are
on incomparable scales. Diverging blue↔red with a gray midpoint, because zero is
a real and meaningful centre here.

In [6]:
region_means = df.groupby("region")[charted_cols].mean().reindex(regions)

# Baseline = the market regions only. REF_REGION is scored against it, not part
# of it, so it can be read as a reference line rather than a ninth data point
# dragging the mean toward itself.
base_means = region_means.loc[baseline_regions]
z = (region_means - base_means.mean()) / base_means.std(ddof=0)
z.columns = charted_names
z = z.fillna(0)

# The colour domain is Z_LIM, not this fork's own max: a per-fork max would give
# the two notebooks different scales for the same SD value, which defeats the
# whole point of standardising. Guard rather than silently clip.
if float(np.abs(z.values).max()) > Z_LIM:
    raise ValueError(
        f"z reaches {np.abs(z.values).max():.2f} SD, past Z_LIM={Z_LIM}. "
        "Raise Z_LIM in BOTH forks so they stay on one scale."
    )

fig = px.imshow(
    z.T,
    labels=dict(x="", y="", color="z-score"),
    color_continuous_scale=DIVERGING,
    zmin=-Z_LIM, zmax=Z_LIM,
    aspect="auto",
    text_auto=".1f",
)
fig.update_traces(
    textfont_size=11,
    hovertemplate="<b>%{x}</b> · %{y}<br>%{z:+.2f} SD vs this fork's regional mean<extra></extra>",
    xgap=2, ygap=2,
)
fig.update_layout(
    title=f"Regional emotion character — {LABEL}<br>"
          f"<sup>Standard deviations from this fork's own regional mean. "
          f"Red = distinctively high, blue = distinctively low. Comparable across forks.</sup>",
    height=60 + 34 * len(charted_names) + 150, width=1000,
    coloraxis_colorbar=dict(title="SD", thickness=12, len=0.7),
    **LAYOUT,
)
fig.show()
display(z.round(2))

,love,sadness,desire,disappointment,approval,curiosity,annoyance,amusement,admiration,confusion,joy,disapproval
region,,,,,,,,,,,,
Argentina,0.13,0.12,-0.27,1.06,-1.42,-0.71,-0.04,0.22,1.26,0.16,-0.52,0.04
Colombia,1.19,0.82,-0.63,-0.37,-0.87,-0.79,-0.28,2.22,1.72,0.22,0.80,-0.58
Global,0.14,0.04,-0.03,1.25,-0.14,0.35,0.29,-0.07,-0.15,0.47,-0.13,0.49
Japan,1.40,1.91,2.22,0.25,0.87,2.00,-1.30,-0.76,-0.92,0.74,2.11,-1.09
Singapore,-0.24,-0.02,0.41,0.79,0.93,0.48,0.15,-0.73,-0.40,-0.94,-0.59,0.58
Spain,-0.20,-0.95,-0.77,-0.61,-0.77,-0.87,0.05,0.31,-0.05,1.13,-0.32,-1.24
Taiwan,-1.82,-1.05,-0.93,-1.99,1.41,-0.71,-0.74,-0.68,-1.07,-1.95,-0.97,0.45
USA,-0.46,-0.82,-0.03,0.87,-0.15,0.61,2.16,-0.58,-0.54,0.65,-0.52,1.84


### 4b. The same z-scores, read by position

The grid above is a fine overview, but colour cannot be decoded to ±0.3
precision — its values are only readable because they're printed into the cells.
Here the identical matrix is encoded by **position** against a shared zero line:
one row per emotion, one dot per region, with colour carrying region *identity*
(the same fixed hues as every other chart in this notebook) instead of magnitude.

Two things this shows that the heatmap hides:

- **Row width is the finding.** z is standardised *across markets within an
  emotion*, so the market dots in every row centre on zero by construction — that
  balance carries no information, but the spread does. A wide row is an emotion
  the markets genuinely split on; a tight row is one they broadly agree about.
  Rows are sorted by that width.
- **Ordering within a row** is read straight off the axis rather than inferred
  from two similar shades.

Both this chart and § 4 above are pinned to a fixed `Z_LIM` shared with the
sibling notebook, so the two forks' versions can be put side by side and read as
one scale.

Three regions are labelled per row: the **low** and **high** market poles, and
**`Global`**, the held-out reference. Global is drawn above its dot rather than
beside it, because it lands mid-row where the horizontal space is already taken.
Note that the poles and the row sort are computed over the markets only — Global
is charted and labelled but never defines the spread, for the same reason it is
kept out of the baseline. The remaining regions are in the legend, the hover, and
the table underneath.

In [7]:
# Rows ordered by how much the *markets* disagree — the widest row goes on top.
# REF_REGION is excluded here for the same reason it is excluded from the
# baseline: it is a reference, so it should not define the spread it is read against.
spread = (z.loc[baseline_regions].max() - z.loc[baseline_regions].min()).sort_values()
row_order = spread.index.tolist()

fig = go.Figure()

# Alternating row bands, so an eye tracking a row left→right doesn't slip off it.
for i in range(0, len(row_order), 2):
    fig.add_shape(type="rect", xref="paper", yref="y", x0=0, x1=1,
                  y0=i - 0.5, y1=i + 0.5, fillcolor="#f7f6f4",
                  line_width=0, layer="below")

for region in regions:
    fig.add_trace(go.Scatter(
        x=z.loc[region, row_order].values,
        y=row_order,
        mode="markers",
        name=region,
        # 2px surface ring, not a border: it keeps overlapping dots countable.
        marker=dict(size=11, color=REGION_COLOR[region],
                    line=dict(width=2, color="white")),
        hovertemplate=f"<b>{region}</b> · %{{y}}<br>%{{x:+.2f}} SD<extra></extra>",
    ))

# Three direct labels per row — the two market poles and the reference region.
# Everyone else is carried by the legend, the hover and the table below; a label
# on all eight would be unreadable.
lo = z.loc[baseline_regions, row_order].idxmin()
hi = z.loc[baseline_regions, row_order].idxmax()

# Poles: outside the dot, on the side the dot points to, so they never sit on data.
fig.add_trace(go.Scatter(
    x=[z.loc[lo[e], e] for e in row_order] + [z.loc[hi[e], e] for e in row_order],
    y=row_order + row_order,
    mode="text",
    text=[lo[e] + "  " for e in row_order] + ["  " + hi[e] for e in row_order],
    textposition=["middle left"] * len(row_order) + ["middle right"] * len(row_order),
    textfont=dict(size=11, color="#6b6862"),
    showlegend=False, hoverinfo="skip", cliponaxis=False,
))

# Reference region: its own lane above the dots, as annotations rather than
# scatter text. Scatter's "top center" offsets by only the marker radius, so
# where the reference sits on top of a pole the two labels collide; a fixed
# pixel yshift clears the pole labels in every row regardless of the values.
if REF_REGION:
    for e in row_order:
        fig.add_annotation(
            x=z.loc[REF_REGION, e], y=e, text=REF_REGION,
            showarrow=False, yshift=17, xanchor="center",
            font=dict(size=10, color="#6b6862"),
        )

fig.update_layout(
    title=f"Regional character per emotion — {LABEL}<br>"
          f"<sup>Each dot is one region's z-score; 0 = this fork's own market baseline"
          + (f", which excludes {REF_REGION}.<br>" if REF_REGION else ".<br>")
          + f"Axis fixed at ±{Z_LIM} SD, so this chart stacks directly against the other fork's. "
            f"Rows sorted by how much the markets disagree.</sup>",
    # The zero rule is the axis zeroline, not an added shape: a shape at x=0 draws
    # alongside the x=0 gridline and renders as a doubled line.
    xaxis=dict(range=[-Z_LIM, Z_LIM], gridcolor="#eceae5", dtick=1,
               zeroline=True, zerolinecolor="#8a8681", zerolinewidth=1.5,
               title=dict(text="standard deviations from this fork's market baseline",
                          font=dict(size=12))),
    yaxis=dict(categoryorder="array", categoryarray=row_order,
               showgrid=False, title=None),
    height=56 * len(row_order) + 210, width=1000,
    legend=dict(orientation="h", y=-0.14, title=None),
    **LAYOUT,
)
fig.show()

# Table twin: the three labelled values per row, plus the width the chart sorts on.
display(
    pd.DataFrame({
        "market spread (SD)": spread,
        "lowest": lo,
        "lowest z": [z.loc[lo[e], e] for e in row_order],
        "highest": hi,
        "highest z": [z.loc[hi[e], e] for e in row_order],
        f"{REF_REGION} z": [z.loc[REF_REGION, e] for e in row_order]
                           if REF_REGION else None,
    }).sort_values("market spread (SD)", ascending=False).round(2)
)

,market spread (SD),lowest,lowest z,highest,highest z,Global z
annoyance,3.46,Japan,-1.30,USA,2.16,0.29
love,3.22,Taiwan,-1.82,Japan,1.40,0.14
desire,3.15,Taiwan,-0.93,Japan,2.22,-0.03
disapproval,3.08,Spain,-1.24,USA,1.84,0.49
confusion,3.08,Taiwan,-1.95,Spain,1.13,0.47
joy,3.07,Taiwan,-0.97,Japan,2.11,-0.13
disappointment,3.04,Taiwan,-1.99,Argentina,1.06,1.25
amusement,2.99,Japan,-0.76,Colombia,2.22,-0.07
sadness,2.96,Taiwan,-1.05,Japan,1.91,0.04
curiosity,2.87,Spain,-0.87,Japan,2.00,0.35


### 5. Where each emotion peaks

Top 5 regions per emotion. Read the *ordering* of the bars; the heights are on
the fork-local scale again.

In [8]:
n = len(charted_cols)
ncols = 3
nrows = -(-n // ncols)

fig = make_subplots(
    rows=nrows, cols=ncols,
    subplot_titles=[c.replace("emotion_", "") for c in charted_cols],
    vertical_spacing=0.09 if nrows <= 4 else 0.05,
    horizontal_spacing=0.07,
)

for idx, col in enumerate(charted_cols):
    r, c = idx // ncols + 1, idx % ncols + 1
    top = df.groupby("region")[col].mean().sort_values(ascending=False).head(5)
    fig.add_trace(
        go.Bar(
            x=top.index, y=top.values,
            marker=dict(color=[REGION_COLOR[i] for i in top.index], cornerradius=4),
            hovertemplate="<b>%{x}</b><br>mean %{y:.3f}<extra></extra>",
            showlegend=False,
        ),
        row=r, col=c,
    )

fig.update_annotations(font_size=13)
fig.update_yaxes(gridcolor="#eceae5", zeroline=False, title=None)
fig.update_xaxes(showgrid=False, tickangle=-35, tickfont_size=10)
fig.update_layout(
    title_text=f"Top 5 regions per emotion — {LABEL}<br>"
               f"<sup>Fork-local mean scores; region colours are fixed across every chart here.</sup>",
    height=260 * nrows + 120, width=1150, bargap=0.35, **LAYOUT,
)
fig.show()

### 6. Score spread within each region

Means hide bimodality — a region can average mid on an emotion because every
song is mid, or because half its songs are extreme. Box plots separate those.

In [9]:
melted = (
    df[["region"] + charted_cols]
      .melt(id_vars="region", var_name="emotion", value_name="score")
)
melted["emotion"] = melted["emotion"].str.replace("emotion_", "", regex=False)

fig = px.box(
    melted, x="emotion", y="score", color="region",
    color_discrete_map=REGION_COLOR,
    category_orders={"emotion": charted_names, "region": regions},
    points=False,
)
fig.update_traces(line_width=1.5, marker_size=4)
fig.update_layout(
    title=f"Score distribution by emotion and region — {LABEL}<br>"
          f"<sup>Box = IQR, line = median. Fork-local scale.</sup>",
    xaxis_title=None, yaxis_title="score",
    boxmode="group", height=560, width=max(1000, 78 * len(charted_names)),
    legend=dict(orientation="h", y=-0.28, title=None),
    **LAYOUT,
)
fig.update_yaxes(gridcolor="#eceae5", zeroline=False)
fig.update_xaxes(showgrid=False, tickangle=-30)
fig.show()

### 7. Table view

Every chart above has a numeric counterpart here — required so nothing in this
notebook is readable by colour alone.

In [10]:
summary = pd.concat(
    {
        "mean score": df.groupby("region")[charted_cols].mean().reindex(regions).T,
    },
    axis=1,
)
summary.index = charted_names
display(summary.round(3))

print("\nGlobal ranking of emotions in this fork:")
display(
    df[emotion_cols].mean().sort_values(ascending=False)
      .rename("mean score").to_frame().round(3)
)

mean score                                                 \
region          Argentina Colombia Global  Japan Singapore  Spain Taiwan   
love                0.219    0.277  0.219  0.289     0.199  0.201  0.111   
sadness             0.101    0.113  0.099  0.131     0.098  0.082  0.081   
desire              0.065    0.054  0.072  0.136     0.084  0.051  0.046   
disappointment      0.078    0.065  0.079  0.070     0.075  0.062  0.050   
approval            0.046    0.052  0.059  0.069     0.070  0.053  0.075   
curiosity           0.050    0.049  0.062  0.081     0.063  0.048  0.050   
annoyance           0.050    0.046  0.054  0.033     0.052  0.051  0.040   
amusement           0.051    0.117  0.041  0.018     0.019  0.054  0.021   
admiration          0.050    0.054  0.038  0.031     0.036  0.039  0.030   
confusion           0.035    0.035  0.036  0.038     0.028  0.040  0.022   
joy                 0.027    0.041  0.031  0.055     0.026  0.029  0.022   
disapproval         0.030    0.028  0.031  0.026     0.031  0.026  0.031   

                       
region            USA  
love            0.186  
sadness         0.085  
desire          0.072  
disappointment  0.076  
approval        0.059  
curiosity       0.065  
annoyance       0.079  
amusement       0.024  
admiration      0.035  
confusion       0.037  
joy             0.027  
disapproval     0.035


Global ranking of emotions in this fork:


,mean score
emotion_love,0.215
emotion_sadness,0.099
emotion_desire,0.073
emotion_disappointment,0.070
emotion_approval,0.060
emotion_curiosity,0.059
emotion_annoyance,0.051
emotion_amusement,0.044
emotion_admiration,0.039
emotion_confusion,0.034


### 8. The neutral label — GoEmotions only

No zero-shot equivalent exists, so this section has no counterpart in `06.1`.
A high neutral share means the classifier read a lot of these lyrics as
affectively flat, which is itself a finding about the taxonomy fit: GoEmotions
was trained on Reddit comments, not song lyrics.

In [11]:
neutral_by_region = (
    df.groupby("region")
      .agg(mean_neutral=("emotion_neutral", "mean"),
           pct_dominant_neutral=("dominant_emotion",
                                 lambda s: (s == "neutral").mean() * 100))
      .reindex(regions)
)

fig = go.Figure(go.Bar(
    x=neutral_by_region.index,
    y=neutral_by_region["pct_dominant_neutral"],
    marker=dict(color=[REGION_COLOR[r] for r in neutral_by_region.index],
                cornerradius=4),
    text=neutral_by_region["pct_dominant_neutral"].round(1),
    texttemplate="%{text}%", textposition="outside",
    hovertemplate="<b>%{x}</b><br>%{y:.1f}% of songs read as neutral<extra></extra>",
))
fig.update_layout(
    title="Share of songs GoEmotions calls <b>neutral</b><br>"
          "<sup>No zero-shot counterpart — 04.1's taxonomy has no neutral label</sup>",
    yaxis_title="% of region's songs", xaxis_title=None,
    height=430, width=900, bargap=0.35, **LAYOUT,
)
fig.update_yaxes(gridcolor="#eceae5", zeroline=False)
fig.update_xaxes(showgrid=False)
fig.show()

display(neutral_by_region.round(2))

,mean_neutral,pct_dominant_neutral
region,,
Argentina,0.26,35.96
Colombia,0.21,25.53
Global,0.24,31.12
Japan,0.20,23.40
Singapore,0.28,33.33
Spain,0.30,41.97
Taiwan,0.48,56.38
USA,0.26,34.69
